# RAG chatbot on Vietnamese Wikipedia (UIT-ViQuAD 2.0)

**Stack:** LangChain + FAISS + `BAAI/bge-m3` embeddings + **Qwen2.5 7B** served by **Ollama**.

**What this notebook does**
1. Installs Ollama and pulls `qwen2.5:7b` (Q4_K_M, ~4.7 GB) on the Kaggle T4 GPU.
2. Builds a corpus from the 3,171 unique paragraphs of the ViQuAD 2.0 dev set, splits them into chunks and indexes them in FAISS.
3. Measures retrieval (Hit@k, MRR) on **all** answerable dev questions.
4. Answers **50 random questions** (seed 42) twice: **with RAG** and **without RAG** (same LLM, same prompt style, only the context changes).
5. Scores answers with EM, F1, Contains and an LLM judge (correctness 1-5, faithfulness 1-5).

Settings: Kaggle GPU **T4**, Internet **on**, dataset `gamind/uit-viquad-2-0` attached.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-ollama faiss-cpu 2>&1 | grep -v -i warning | tail -2

## 1. Install Ollama

In [ ]:
!apt-get update -qq > /dev/null && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1; tail -2 /tmp/ollama_install.log
!ollama --version

## 2. Project code (`src/`)
These cells are generated from the repo's `src/` folder.

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [ ]:
%%writefile src/__init__.py
# package

In [ ]:
%%writefile src/config.py
"""All settings in one place. Change a value here, not inside the code."""
from dataclasses import dataclass


@dataclass
class Config:
    # data
    viquad_dev: str = "/kaggle/input/uit-viquad-2-0/dev.json"  # Kaggle dataset gamind/uit-viquad-2-0
    n_questions: int = 50
    seed: int = 42

    # chunking
    chunk_size: int = 500
    chunk_overlap: int = 100
    add_title: bool = True  # prepend the Wikipedia article title to every chunk

    # retrieval
    embed_model: str = "BAAI/bge-m3"
    top_k: int = 4
    max_k_curve: int = 10  # Hit@k curve is computed for k = 1..max_k_curve

    # generation (Ollama)
    llm_model: str = "qwen2.5:7b"  # Q4_K_M, ~4.7 GB
    ollama_url: str = "http://localhost:11434"
    temperature: float = 0.0
    num_ctx: int = 4096
    num_predict: int = 128

    # judge (same local model, see README "Limitations")
    judge_model: str = "qwen2.5:7b"

    out_dir: str = "results"

In [ ]:
%%writefile src/data.py
"""Load UIT-ViQuAD 2.0, build the document corpus and the 50-question test set."""
import glob
import json
import os
import random

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def find_dev_json(path):
    """Use `path` if it exists, else search /kaggle/input (mount paths differ between images)."""
    if os.path.exists(path):
        return path
    hits = sorted(glob.glob("/kaggle/input/**/dev.json", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{path} not found and no dev.json under /kaggle/input")
    return hits[0]


def load_viquad(path):
    """dev.json is a flat list of {id, title, context, question, answers{text, answer_start}}."""
    path = find_dev_json(path)
    with open(path, encoding="utf-8") as f:
        return json.load(f)["data"]


def build_corpus(records):
    """One Document per unique paragraph. Returns (docs, context -> ctx_id map)."""
    ctx_ids, docs = {}, []
    for r in records:
        if r["context"] not in ctx_ids:
            ctx_ids[r["context"]] = len(docs)
            docs.append(Document(page_content=r["context"],
                                 metadata={"ctx_id": len(docs), "title": r["title"]}))
    return docs, ctx_ids


def answerable_questions(records, ctx_ids):
    """Questions that have a gold answer (ViQuAD 2.0 also has unanswerable ones)."""
    out = []
    for r in records:
        if r["answers"]["text"]:
            out.append({"id": r["id"], "title": r["title"], "question": r["question"],
                        "answers": sorted(set(r["answers"]["text"])),
                        "ctx_id": ctx_ids[r["context"]]})
    return out


def sample_questions(questions, n, seed):
    qs = sorted(questions, key=lambda q: q["id"])
    return random.Random(seed).sample(qs, n)


def split_docs(docs, chunk_size, chunk_overlap, add_title=True):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", ", ", " ", ""])
    chunks = splitter.split_documents(docs)
    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = i
        if add_title:
            c.page_content = f"{c.metadata['title']}: {c.page_content}"
    return chunks

In [ ]:
%%writefile src/rag.py
"""The RAG chatbot: bge-m3 embeddings + FAISS + Qwen2.5 (Ollama), wired with LangChain."""
import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

SYSTEM = ("Bạn là trợ lý hỏi đáp tiếng Việt. Trả lời thật ngắn gọn, chỉ một cụm từ "
          "hoặc một câu ngắn, không giải thích.")

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM + " Chỉ dùng thông tin trong NGỮ CẢNH. "
               "Nếu ngữ cảnh không có câu trả lời, hãy nói: Không biết."),
    ("human", "NGỮ CẢNH:\n{context}\n\nCÂU HỎI: {question}\nTRẢ LỜI:"),
])

NO_RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM + " Nếu không chắc, hãy nói: Không biết."),
    ("human", "CÂU HỎI: {question}\nTRẢ LỜI:"),
])


def get_embeddings(model_name, device="cuda"):
    return HuggingFaceEmbeddings(model_name=model_name, model_kwargs={"device": device},
                                 encode_kwargs={"normalize_embeddings": True, "batch_size": 32})


def build_index(chunks, embeddings):
    # vectors are L2-normalised, so L2 ranking == cosine ranking
    return FAISS.from_documents(chunks, embeddings)


def get_llm(cfg, model=None, json_mode=False):
    return ChatOllama(model=model or cfg.llm_model, base_url=cfg.ollama_url,
                      temperature=cfg.temperature, num_ctx=cfg.num_ctx,
                      num_predict=cfg.num_predict, format="json" if json_mode else None)


def format_context(docs):
    return "\n\n".join(f"[{i + 1}] {d.page_content}" for i, d in enumerate(docs))


class RAGChatbot:
    def __init__(self, index, llm, k=4):
        self.index, self.k = index, k
        self.rag_chain = RAG_PROMPT | llm | StrOutputParser()
        self.plain_chain = NO_RAG_PROMPT | llm | StrOutputParser()

    def retrieve(self, question, k=None):
        return self.index.similarity_search(question, k=k or self.k)

    def ask(self, question):
        docs = self.retrieve(question)
        answer = self.rag_chain.invoke({"context": format_context(docs), "question": question})
        return {"answer": answer.strip(), "docs": docs}

    def ask_no_rag(self, question):
        return {"answer": self.plain_chain.invoke({"question": question}).strip(), "docs": []}


def batch_retrieve_ctx_ids(index, embeddings, questions, k):
    """Fast retrieval for many questions at once: returns ctx_id lists (top-k chunks)."""
    q = np.asarray(embeddings.embed_documents(questions), dtype="float32")
    _, ids = index.index.search(q, k)
    store, id_map = index.docstore, index.index_to_docstore_id
    return [[store.search(id_map[i]).metadata["ctx_id"] for i in row if i >= 0] for row in ids]

In [ ]:
%%writefile src/metrics.py
"""Answer metrics (SQuAD-style EM/F1 adapted to Vietnamese) and retrieval metrics."""
import re
import string
import unicodedata
from collections import Counter

_PUNCT = set(string.punctuation) | set("“”‘’–—…«»")


def normalize(text):
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = "".join(" " if ch in _PUNCT else ch for ch in text)
    return re.sub(r"\s+", " ", text).strip()


def exact_match(pred, golds):
    p = normalize(pred)
    return float(any(p == normalize(g) for g in golds))


def f1(pred, golds):
    """Token F1 over Vietnamese syllables (space-separated), best over gold answers."""
    p = normalize(pred).split()
    best = 0.0
    for g in golds:
        g = normalize(g).split()
        common = sum((Counter(p) & Counter(g)).values())
        if common:
            pr, rc = common / len(p), common / len(g)
            best = max(best, 2 * pr * rc / (pr + rc))
    return best


def contains(pred, golds):
    """1 if a gold answer appears inside the prediction (LLMs often add a few words)."""
    p = normalize(pred)
    return float(any(normalize(g) in p for g in golds))


def hit_at_k(retrieved_ctx, gold_ctx, k):
    return float(gold_ctx in retrieved_ctx[:k])


def reciprocal_rank(retrieved_ctx, gold_ctx, k):
    for rank, c in enumerate(retrieved_ctx[:k], 1):
        if c == gold_ctx:
            return 1.0 / rank
    return 0.0

In [ ]:
%%writefile src/judge.py
"""LLM-as-judge with the local model: correctness (vs gold) and faithfulness (vs context)."""
import json
import re

from langchain_core.prompts import ChatPromptTemplate

CORRECTNESS = ChatPromptTemplate.from_messages([
    ("system", "You grade answers to Vietnamese questions. Compare the MODEL ANSWER with the "
               "GOLD ANSWER. Score 1-5: 5 = same meaning as gold, 4 = correct but with extra or "
               "missing minor detail, 3 = partly correct, 2 = mostly wrong, 1 = wrong or "
               "'Không biết'. Reply only JSON: {{\"score\": <1-5>, \"reason\": \"<short>\"}}"),
    ("human", "QUESTION: {question}\nGOLD ANSWER: {gold}\nMODEL ANSWER: {answer}"),
])

FAITHFULNESS = ChatPromptTemplate.from_messages([
    ("system", "You check if an answer is supported by the given CONTEXT. Score 1-5: "
               "5 = fully supported, 3 = partly supported, 1 = not supported or invented. "
               "If the answer is 'Không biết', score 5 only if the context really lacks the "
               "answer, else 1. Reply only JSON: {{\"score\": <1-5>, \"reason\": \"<short>\"}}"),
    ("human", "CONTEXT:\n{context}\n\nQUESTION: {question}\nANSWER: {answer}"),
])


def parse_score(text):
    try:
        s = int(json.loads(text)["score"])
    except Exception:
        m = re.search(r"[1-5]", text or "")
        s = int(m.group()) if m else 1
    return min(max(s, 1), 5)


class Judge:
    def __init__(self, llm_json):
        self.correct_chain = CORRECTNESS | llm_json
        self.faith_chain = FAITHFULNESS | llm_json

    def correctness(self, question, golds, answer):
        out = self.correct_chain.invoke({"question": question, "gold": " | ".join(golds),
                                         "answer": answer})
        return parse_score(out.content)

    def faithfulness(self, question, context, answer):
        out = self.faith_chain.invoke({"question": question, "context": context, "answer": answer})
        return parse_score(out.content)

In [ ]:
%%writefile src/ollama_utils.py
"""Start a local Ollama server and pull a model (used by the Kaggle notebook)."""
import json
import subprocess
import time
import urllib.request


def server_up(url="http://localhost:11434"):
    try:
        urllib.request.urlopen(f"{url}/api/tags", timeout=2)
        return True
    except Exception:
        return False


def start_server(log_path="ollama.log", url="http://localhost:11434", timeout=60):
    if server_up(url):
        return None
    proc = subprocess.Popen(["ollama", "serve"], stdout=open(log_path, "w"),
                            stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(timeout):
        if server_up(url):
            return proc
        time.sleep(1)
    raise RuntimeError(f"Ollama did not start, see {log_path}")


def pull(model):
    subprocess.run(["ollama", "pull", model], check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)


def model_info(model, url="http://localhost:11434"):
    req = urllib.request.Request(f"{url}/api/show", data=json.dumps({"model": model}).encode(),
                                 headers={"Content-Type": "application/json"})
    d = json.load(urllib.request.urlopen(req))
    return d.get("details", {})

In [ ]:
%%writefile src/evaluate.py
"""Full pipeline: corpus -> FAISS -> retrieval eval -> RAG vs no-RAG answers -> scores.

Usage (needs a running Ollama server with the model pulled):
    python -m src.evaluate --viquad_dev path/to/dev.json --out_dir results
"""
import argparse
import json
import os
import time

import pandas as pd
from tqdm.auto import tqdm

from .config import Config
from .data import answerable_questions, build_corpus, load_viquad, sample_questions, split_docs
from .judge import Judge
from .metrics import contains, exact_match, f1, hit_at_k, reciprocal_rank
from .rag import (RAGChatbot, batch_retrieve_ctx_ids, build_index, format_context,
                  get_embeddings, get_llm)


def prepare(cfg):
    records = load_viquad(cfg.viquad_dev)
    docs, ctx_ids = build_corpus(records)
    all_q = answerable_questions(records, ctx_ids)
    test_q = sample_questions(all_q, cfg.n_questions, cfg.seed)
    chunks = split_docs(docs, cfg.chunk_size, cfg.chunk_overlap, cfg.add_title)
    stats = {"paragraphs": len(docs), "chunks": len(chunks),
             "answerable_questions": len(all_q), "test_questions": len(test_q)}
    return docs, chunks, all_q, test_q, stats


def retrieval_eval(index, emb, questions, max_k):
    """Hit@k and MRR on ALL answerable dev questions (cheap: no LLM)."""
    t = time.time()
    got = batch_retrieve_ctx_ids(index, emb, [q["question"] for q in questions], max_k)
    curve = [{"k": k, "hit": sum(hit_at_k(r, q["ctx_id"], k) for r, q in zip(got, questions))
              / len(questions)} for k in range(1, max_k + 1)]
    mrr = {f"MRR@{k}": sum(reciprocal_rank(r, q["ctx_id"], k) for r, q in zip(got, questions))
           / len(questions) for k in (4, max_k)}
    return pd.DataFrame(curve), mrr, time.time() - t


def answer_eval(cfg, bot, judge, test_q):
    rows = []
    for q in tqdm(test_q, desc="answering"):
        docs = bot.retrieve(q["question"])
        ctx = format_context(docs)
        retrieved = [d.metadata["ctx_id"] for d in docs]
        t0 = time.time()
        rag = bot.rag_chain.invoke({"context": ctx, "question": q["question"]}).strip()
        t1 = time.time()
        plain = bot.plain_chain.invoke({"question": q["question"]}).strip()
        t2 = time.time()
        rows.append({
            "id": q["id"], "title": q["title"], "question": q["question"],
            "gold": " | ".join(q["answers"]),
            "hit@4": hit_at_k(retrieved, q["ctx_id"], cfg.top_k),
            "rr@4": reciprocal_rank(retrieved, q["ctx_id"], cfg.top_k),
            "rag_answer": rag, "norag_answer": plain,
            "rag_em": exact_match(rag, q["answers"]), "norag_em": exact_match(plain, q["answers"]),
            "rag_f1": f1(rag, q["answers"]), "norag_f1": f1(plain, q["answers"]),
            "rag_contains": contains(rag, q["answers"]),
            "norag_contains": contains(plain, q["answers"]),
            "rag_judge": judge.correctness(q["question"], q["answers"], rag),
            "norag_judge": judge.correctness(q["question"], q["answers"], plain),
            "rag_faithfulness": judge.faithfulness(q["question"], ctx, rag),
            "rag_sec": round(t1 - t0, 2), "norag_sec": round(t2 - t1, 2),
            "sources": "; ".join(f"{d.metadata['title']}#{d.metadata['ctx_id']}" for d in docs),
        })
    return pd.DataFrame(rows)


def summarize(df):
    s = {"hit@4": df["hit@4"].mean(), "MRR@4": df["rr@4"].mean()}
    for mode in ("rag", "norag"):
        s[mode] = {"EM": df[f"{mode}_em"].mean(), "F1": df[f"{mode}_f1"].mean(),
                   "Contains": df[f"{mode}_contains"].mean(),
                   "Judge(1-5)": df[f"{mode}_judge"].mean(),
                   "Judge>=4": (df[f"{mode}_judge"] >= 4).mean(),
                   "sec/answer": df[f"{mode}_sec"].mean()}
    s["rag"]["Faithfulness(1-5)"] = df["rag_faithfulness"].mean()
    # RAG answers split by whether retrieval found the gold paragraph
    for hit, name in ((1.0, "rag_when_hit"), (0.0, "rag_when_miss")):
        sub = df[df["hit@4"] == hit]
        s[name] = {"n": len(sub), "F1": sub["rag_f1"].mean() if len(sub) else None,
                   "Judge(1-5)": sub["rag_judge"].mean() if len(sub) else None}
    return json.loads(json.dumps(s, default=float))


def run(cfg):
    os.makedirs(cfg.out_dir, exist_ok=True)
    docs, chunks, all_q, test_q, stats = prepare(cfg)
    print("data:", stats)
    with open(os.path.join(cfg.out_dir, "questions_50.json"), "w", encoding="utf-8") as f:
        json.dump(test_q, f, ensure_ascii=False, indent=1)

    t = time.time()
    emb = get_embeddings(cfg.embed_model)
    index = build_index(chunks, emb)
    stats["index_sec"] = round(time.time() - t, 1)

    curve, mrr, sec = retrieval_eval(index, emb, all_q, cfg.max_k_curve)
    curve.to_csv(os.path.join(cfg.out_dir, "retrieval_curve.csv"), index=False)

    bot = RAGChatbot(index, get_llm(cfg), k=cfg.top_k)
    judge = Judge(get_llm(cfg, model=cfg.judge_model, json_mode=True))
    df = answer_eval(cfg, bot, judge, test_q)
    df.to_csv(os.path.join(cfg.out_dir, "predictions.csv"), index=False)

    summary = {"config": vars(cfg), "data": stats,
               "retrieval_all_dev": {**{f"Hit@{r.k}": r.hit for r in curve.itertuples()}, **mrr,
                                     "n": len(all_q), "sec": round(sec, 1)},
               "test50": summarize(df)}
    with open(os.path.join(cfg.out_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    return index, emb, bot, df, curve, summary


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    for k, v in vars(Config()).items():
        ap.add_argument(f"--{k}", type=type(v) if not isinstance(v, bool) else lambda x: x == "1",
                        default=v)
    run(Config(**vars(ap.parse_args())))

In [ ]:
%%writefile src/plots.py
"""Figures for the README."""
import os

import matplotlib.pyplot as plt

BLUE, GREY = "#2a6fdb", "#9aa5b1"


def plot_rag_vs_norag(summary, out_dir):
    s = summary["test50"]
    names = ["EM", "F1", "Contains", "Judge>=4"]
    rag = [s["rag"][n] * 100 for n in names]
    plain = [s["norag"][n] * 100 for n in names]
    x = range(len(names))
    fig, ax = plt.subplots(figsize=(7, 4))
    b1 = ax.bar([i - 0.2 for i in x], plain, 0.4, label="No RAG (Qwen2.5 7B only)", color=GREY)
    b2 = ax.bar([i + 0.2 for i in x], rag, 0.4, label="RAG (bge-m3 + FAISS, top-4)", color=BLUE)
    ax.bar_label(b1, fmt="%.0f", fontsize=9)
    ax.bar_label(b2, fmt="%.0f", fontsize=9)
    ax.set_xticks(list(x), names)
    ax.set_ylabel("% of 50 questions")
    ax.set_ylim(0, 105)
    ax.set_title("Answer quality on 50 UIT-ViQuAD 2.0 questions")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    path = os.path.join(out_dir, "rag_vs_norag.png")
    fig.savefig(path, dpi=150)
    return path


def plot_hit_curve(curve, n, out_dir):
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(curve["k"], curve["hit"] * 100, marker="o", color=BLUE)
    for k, h in zip(curve["k"], curve["hit"]):
        if k in (1, 4, curve["k"].max()):
            ax.annotate(f"{h * 100:.1f}", (k, h * 100), textcoords="offset points",
                        xytext=(0, 7), ha="center", fontsize=9)
    ax.set_xlabel("k (retrieved chunks)")
    ax.set_ylabel("Hit@k (%)")
    ax.set_title(f"Retrieval: gold paragraph in top-k ({n} dev questions)")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    path = os.path.join(out_dir, "retrieval_hit_at_k.png")
    fig.savefig(path, dpi=150)
    return path

## 3. Start Ollama and pull Qwen2.5 7B

In [ ]:
import time
from src.config import Config
from src.ollama_utils import start_server, pull, model_info

cfg = Config()
server = start_server("results/ollama.log")
t = time.time(); pull(cfg.llm_model)
print(f"pulled {cfg.llm_model} in {time.time() - t:.0f}s:", model_info(cfg.llm_model))

## 4. Data: corpus, chunks and the 50 test questions

In [ ]:
import json, pandas as pd
from src.evaluate import prepare

docs, chunks, all_q, test_q, stats = prepare(cfg)
print(stats)
json.dump(test_q, open("results/questions_50.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("example chunk:\n", chunks[0].page_content[:300])
pd.DataFrame(test_q)[["title", "question", "answers"]].head(10)

## 5. Embeddings (bge-m3) + FAISS index

In [ ]:
from src.rag import get_embeddings, build_index

t = time.time()
emb = get_embeddings(cfg.embed_model)
index = build_index(chunks, emb)
stats["index_sec"] = round(time.time() - t, 1)
print(f"indexed {index.index.ntotal} chunks in {stats['index_sec']}s")

## 6. Retrieval quality on all answerable dev questions
A hit means a retrieved chunk comes from the paragraph that holds the gold answer.

In [ ]:
from src.evaluate import retrieval_eval

curve, mrr, sec = retrieval_eval(index, emb, all_q, cfg.max_k_curve)
curve.to_csv("results/retrieval_curve.csv", index=False)
print(f"{len(all_q)} questions in {sec:.0f}s", {k: round(v, 4) for k, v in mrr.items()})
curve.round(4)

## 7. Try the chatbot

In [ ]:
from src.rag import RAGChatbot, get_llm

bot = RAGChatbot(index, get_llm(cfg), k=cfg.top_k)
for q in ["Thủ đô của Nhật Bản là thành phố nào?",
          "Bức tường Berlin được xây dựng vào năm nào?",
          test_q[0]["question"]]:
    out = bot.ask(q)
    print("Q:", q)
    print("A:", out["answer"])
    print("sources:", [d.metadata["title"] for d in out["docs"]], "\n")

## 8. Evaluate 50 questions: RAG vs no RAG

In [ ]:
from src.judge import Judge
from src.evaluate import answer_eval, summarize

judge = Judge(get_llm(cfg, model=cfg.judge_model, json_mode=True))
t = time.time()
df = answer_eval(cfg, bot, judge, test_q)
print(f"done in {time.time() - t:.0f}s")
df.to_csv("results/predictions.csv", index=False)
df[["question", "gold", "rag_answer", "norag_answer", "hit@4", "rag_judge", "norag_judge"]].head(15)

## 9. Results

In [ ]:
from src.plots import plot_rag_vs_norag, plot_hit_curve

summary = {"config": vars(cfg), "data": stats,
           "retrieval_all_dev": {**{f"Hit@{r.k}": r.hit for r in curve.itertuples()}, **mrr,
                                 "n": len(all_q), "sec": round(sec, 1)},
           "test50": summarize(df)}
json.dump(summary, open("results/metrics.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
plot_rag_vs_norag(summary, "results"); plot_hit_curve(curve, len(all_q), "results")

t50 = summary["test50"]
table = pd.DataFrame({m: t50[m] for m in ("rag", "norag")}).round(3)
print(f"Retrieval on 50: Hit@4={t50['hit@4']:.2f}  MRR@4={t50['MRR@4']:.3f}")
table

In [ ]:
from IPython.display import Image, display
display(Image("results/rag_vs_norag.png")); display(Image("results/retrieval_hit_at_k.png"))

## 10. Save environment info and print results to the log

In [ ]:
import subprocess, sys, importlib.metadata as md_
pk = ["langchain-core", "langchain-community", "langchain-huggingface", "langchain-ollama",
      "faiss-cpu", "sentence-transformers", "torch", "transformers"]
env = {p: md_.version(p) for p in pk}
env["python"] = sys.version.split()[0]
env["ollama"] = subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip()
env["gpu"] = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                            capture_output=True, text=True).stdout.strip()
json.dump(env, open("results/environment.json", "w"), indent=2)
print("ENV_JSON", json.dumps(env))
print("METRICS_JSON", json.dumps(summary, ensure_ascii=False))
print("PREDICTIONS_JSON", df.to_json(orient="records", force_ascii=False))